## 쉐포라 데이터로 모델 재설계

In [43]:
import pandas as pd
import numpy as np

df_product = pd.read_csv(r"C:\workspace\finalproject\data\sephora_products.csv")
df_ingredient = pd.read_csv(r"C:\workspace\finalproject\data\sephora_ingredients.csv")
df_sentiment = pd.read_csv(r"C:\workspace\finalproject\data\sephora_reviews_with_sentiment.csv")

print(f"상품: {df_product.shape}")
print(f"성분: {df_ingredient.shape}")
print(f"감성: {df_sentiment.shape}")

상품: (1838, 15)
성분: (1802, 118)
감성: (905553, 5)


### Y값 생성 (감성점수 + 리뷰수/평점 → PCA → y_final)

In [44]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, QuantileTransformer

df_product.columns = [
    'product_id', '상품명', '브랜드명', '공급가', '할인율', '할인가',
    '용량', '옵션', '옵션종류', '사용기한', '카테고리대', '카테고리중',
    '리뷰수', '평점', 'target_category'
]

# y1: 제품별 평균 감성점수 (mBERT 기반)
y1 = df_sentiment.groupby('product_id')['sentiment_score'].mean().reset_index()
y1.columns = ['product_id', 'y1']

# y2: 로그 정규화 리뷰수 (리뷰수 많을수록 시장 인기 반영)
df_y = df_product[['product_id', '리뷰수']].merge(y1, on='product_id', how='inner')
df_y = df_y.dropna(subset=['리뷰수', 'y1'])
df_y['y2'] = np.log1p(df_y['리뷰수']) / np.log1p(df_y['리뷰수'].max())

# 상관관계 확인 (평점 제외 근거)
rating_df = df_product[['product_id', '평점']].merge(y1, on='product_id', how='inner')
print(f"y1 vs y2 상관관계: {df_y['y1'].corr(df_y['y2']):.3f}  ← 독립적이면 OK")
print(f"y1 vs 평점 상관관계: {rating_df['y1'].corr(rating_df['평점']):.3f}  ← 높으면 평점=y1 중복")

# PCA (y1 + y2만 사용, 평점 제외)
scaler = StandardScaler()
y_scaled = scaler.fit_transform(df_y[['y1', 'y2']])

pca = PCA(n_components=1)
y_pca = pca.fit_transform(y_scaled)

qt = QuantileTransformer(output_distribution='uniform', random_state=42)
df_y['y_final'] = qt.fit_transform(y_pca)

print(f"\nY 생성 완료: {len(df_y)}개 제품")
print(f"PC1 분산 설명력: {pca.explained_variance_ratio_[0]:.3f}")
print(f"PC1 로딩벡터: y1={pca.components_[0][0]:.3f}, y2={pca.components_[0][1]:.3f}")
print(f"\ny_final 분포:")
print(f"  하위 30% (0.0~0.3): {(df_y['y_final'] < 0.3).mean()*100:.1f}%")
print(f"  중간 40% (0.3~0.7): {((df_y['y_final'] >= 0.3) & (df_y['y_final'] < 0.7)).mean()*100:.1f}%")
print(f"  상위 30% (0.7~1.0): {(df_y['y_final'] >= 0.7).mean()*100:.1f}%")

y1 vs y2 상관관계: 0.164  ← 독립적이면 OK
y1 vs 평점 상관관계: 0.915  ← 높으면 평점=y1 중복

Y 생성 완료: 1801개 제품
PC1 분산 설명력: 0.582
PC1 로딩벡터: y1=0.707, y2=0.707

y_final 분포:
  하위 30% (0.0~0.3): 30.0%
  중간 40% (0.3~0.7): 39.9%
  상위 30% (0.7~1.0): 30.0%


### X 피처: 성분 기반 (K-beauty 대표 성분 / 미국 트렌드 성분)

In [45]:
# ─────────────────────────────────────────────────────────────────────
# K뷰티 대표 성분 상위 10개
# 근거: KOTRA 2024-2025 K뷰티 트렌드 보고서, Mintel Beauty & Personal Care,
#       식약처 기능성 화장품 고시 성분, Grand View Research K-Beauty Market Report
# ─────────────────────────────────────────────────────────────────────
K_BEAUTY_INGREDIENTS = {
    'galactomyces':    'Galactomyces Ferment Filtrate',  # KOTRA K뷰티 발효 성분 1위, SK-II 핵심 성분
    'centella':        'Centella Asiatica',               # KOTRA 2025 K뷰티 트렌드 1위 (Cica 열풍)
    'snail':           'Snail Secretion Filtrate',        # K뷰티 아이콘 성분 (달팽이 뮤신) - 확장: snail secretion/filtrate 모두 포함
    'bifida':          'Bifida Ferment Lysate',           # Mintel 2024 마이크로바이옴 트렌드 - 확장: bifida ferment 모두 포함
    'adenosine':       'Adenosine',                       # 식약처 고시 주름개선 기능성 성분 (한국 독자 규격)
    'oryza sativa':    'Rice Extract',                    # 전통 한국 미용 성분, 미백 임상 연구 다수
    'beta-glucan':     'Beta-Glucan',                     # K뷰티 피부장벽 강화 대표 성분 (Mintel 2024)
    'saccharomyces':   'Saccharomyces Ferment Filtrate',  # K뷰티 발효 열풍 주요 성분
    'lactobacillus':   'Lactobacillus Ferment',           # K뷰티 프로바이오틱 성분 (Mintel 마이크로바이옴)
    'propolis':        'Propolis Extract',                 # COSRX 등 K뷰티 브랜드 대표 천연 성분
}

# ─────────────────────────────────────────────────────────────────────
# 미국 트렌드 성분 상위 10개
# 근거: Google Trends 2024-2025, Statista US Skincare Market,
#       Mintel US Skincare Report 2025, ULTA/Sephora 트렌드 리포트
# ─────────────────────────────────────────────────────────────────────
US_TREND_INGREDIENTS = {
    'niacinamide':     'Niacinamide',      # Google Trends 미국 스킨케어 검색 1위
    'ceramide':        'Ceramide',         # 피부장벽 스테디셀러 (Statista 2024)
    'retinol':         'Retinol',          # 안티에이징 1위, 월 검색 6만건
    'bakuchiol':       'Bakuchiol',        # 2024-2025 천연 레티놀 대체 급부상 (Mintel)
    'peptide':         'Peptide',          # 안티에이징/탄력 트렌드
    'hyaluronic acid': 'Hyaluronic Acid',  # 수분 스테디셀러
    'ascorbic acid':   'Ascorbic Acid',    # 비타민C (검색량 174% 급증, ULTA 리포트)
    'salicylic acid':  'Salicylic Acid',   # 여드름 케어 대표 성분
    'glycolic acid':   'Glycolic Acid',    # AHA 대표 성분
    'azelaic acid':    'Azelaic Acid',     # 2025 미국 급부상 성분 (Mintel US Skincare)
}

# ─────────────────────────────────────────────────────────────────────
# 성분 텍스트 통합 및 이진 피처 생성
# ─────────────────────────────────────────────────────────────────────
ing_cols = [c for c in df_ingredient.columns if c.startswith('성분_')]
df_ingredient['all_ingredients'] = (
    df_ingredient[ing_cols]
    .fillna('')
    .apply(lambda row: ' '.join(row).lower(), axis=1)
)

# K뷰티 성분 이진 피처 (성분 보유 여부 0/1)
for key in K_BEAUTY_INGREDIENTS:
    col = f"k_{key.replace(' ', '_')}"
    df_ingredient[col] = df_ingredient['all_ingredients'].str.contains(key, case=False).astype(int)

# 미국 트렌드 성분 이진 피처 (성분 보유 여부 0/1)
for key in US_TREND_INGREDIENTS:
    col = f"us_{key.replace(' ', '_')}"
    df_ingredient[col] = df_ingredient['all_ingredients'].str.contains(key, case=False).astype(int)

df_ingredient['total_ingredient_count'] = df_ingredient[ing_cols].notna().sum(axis=1)

# ─────────────────────────────────────────────────────────────────────
# 합산 비율 피처: 희박한 이진 피처를 보완하는 집계 신호
# - k_beauty_ratio: K뷰티 대표 성분 보유 비율 (보유 수 / 10)
# - us_trend_ratio: 미국 트렌드 성분 보유 비율 (보유 수 / 10)
# ─────────────────────────────────────────────────────────────────────
k_feat_cols  = [f"k_{k.replace(' ', '_')}"  for k in K_BEAUTY_INGREDIENTS]
us_feat_cols = [f"us_{k.replace(' ', '_')}" for k in US_TREND_INGREDIENTS]

df_ingredient['k_beauty_ratio']  = df_ingredient[k_feat_cols].sum(axis=1)  / len(K_BEAUTY_INGREDIENTS)
df_ingredient['us_trend_ratio']  = df_ingredient[us_feat_cols].sum(axis=1) / len(US_TREND_INGREDIENTS)

# 보유율 확인
SPARSE_THRESHOLD = 0.02  # 2% 미만이면 희박 경고

print("=== K뷰티 성분 피처 보유율 ===")
for col, (key, name) in zip(k_feat_cols, K_BEAUTY_INGREDIENTS.items()):
    cnt  = df_ingredient[col].sum()
    rate = cnt / len(df_ingredient)
    flag = " ⚠ 희박" if rate < SPARSE_THRESHOLD else ""
    print(f"  {name:<35} {cnt:>4}개 ({rate*100:.1f}%){flag}")

print("\n=== 미국 트렌드 성분 피처 보유율 ===")
for col, (key, name) in zip(us_feat_cols, US_TREND_INGREDIENTS.items()):
    cnt  = df_ingredient[col].sum()
    rate = cnt / len(df_ingredient)
    flag = " ⚠ 희박" if rate < SPARSE_THRESHOLD else ""
    print(f"  {name:<35} {cnt:>4}개 ({rate*100:.1f}%){flag}")

print(f"\n=== 합산 비율 피처 분포 ===")
print(f"  k_beauty_ratio  평균: {df_ingredient['k_beauty_ratio'].mean():.3f}  max: {df_ingredient['k_beauty_ratio'].max():.3f}")
print(f"  us_trend_ratio  평균: {df_ingredient['us_trend_ratio'].mean():.3f}  max: {df_ingredient['us_trend_ratio'].max():.3f}")

=== K뷰티 성분 피처 보유율 ===
  Galactomyces Ferment Filtrate         18개 (1.0%) ⚠ 희박
  Centella Asiatica                     74개 (4.1%)
  Snail Secretion Filtrate               3개 (0.2%) ⚠ 희박
  Bifida Ferment Lysate                 23개 (1.3%) ⚠ 희박
  Adenosine                            185개 (10.3%)
  Rice Extract                         137개 (7.6%)
  Beta-Glucan                           55개 (3.1%)
  Saccharomyces Ferment Filtrate       136개 (7.5%)
  Lactobacillus Ferment                195개 (10.8%)
  Propolis Extract                       8개 (0.4%) ⚠ 희박

=== 미국 트렌드 성분 피처 보유율 ===
  Niacinamide                          289개 (16.0%)
  Ceramide                             174개 (9.7%)
  Retinol                               84개 (4.7%)
  Bakuchiol                             53개 (2.9%)
  Peptide                              358개 (19.9%)
  Hyaluronic Acid                      146개 (8.1%)
  Ascorbic Acid                        178개 (9.9%)
  Salicylic Acid                       195개 (10.8%)
  Glycoli

### X + Y 병합 → 최종 모델 데이터셋 생성 (상품명 TF-IDF 포함)

In [53]:
# TF-IDF 어휘 사전 확인 (한국 상품 번역 시 참고용)
vocab = tfidf.get_feature_names_out()
print(f"=== TF-IDF 어휘 사전 ({len(vocab)}개) ===")
print("한국 상품 영문 번역 시 아래 단어들이 포함되면 피처에 반영됩니다.\n")
for i, word in enumerate(sorted(vocab), 1):
    print(f"  {i:>3}. {word}")

=== TF-IDF 어휘 사전 (50개) ===
한국 상품 영문 번역 시 아래 단어들이 포함되면 피처에 반영됩니다.

    1. 30
    2. acid
    3. acne
    4. advanced
    5. aging
    6. aha
    7. anti
    8. anti aging
    9. bha
   10. brightening
   11. broad
   12. broad spectrum
   13. clarifying
   14. cleanser
   15. cleansing
   16. cream
   17. daily
   18. dark
   19. exfoliating
   20. eye
   21. eye cream
   22. face
   23. facial
   24. firming
   25. gel
   26. glow
   27. hyaluronic
   28. hyaluronic acid
   29. hydrating
   30. lotion
   31. mask
   32. mini
   33. moisturizer
   34. niacinamide
   35. night
   36. oil
   37. pore
   38. repair
   39. retinol
   40. serum
   41. skin
   42. spectrum
   43. spectrum spf
   44. spf
   45. spf 30
   46. sunscreen
   47. toner
   48. treatment
   49. vitamin
   50. water


In [46]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

k_feat_cols  = [f"k_{k.replace(' ', '_')}"  for k in K_BEAUTY_INGREDIENTS if k != 'snail']
us_feat_cols = [f"us_{k.replace(' ', '_')}" for k in US_TREND_INGREDIENTS]
feat_cols = k_feat_cols + us_feat_cols + ['total_ingredient_count', 'k_beauty_ratio', 'us_trend_ratio']

df_feat = df_ingredient[['product_id'] + feat_cols].copy()

# 용량 파싱 (mL 우선, 없으면 oz → mL 변환)
def parse_volume(text):
    if pd.isna(text):
        return np.nan
    ml = re.search(r'(\d+\.?\d*)\s*mL', str(text), re.IGNORECASE)
    if ml:
        return float(ml.group(1))
    oz = re.search(r'(\d+\.?\d*)\s*oz', str(text), re.IGNORECASE)
    if oz:
        return round(float(oz.group(1)) * 29.57, 1)
    return np.nan

df_product['용량_ml'] = df_product['용량'].apply(parse_volume)
vol_median = df_product['용량_ml'].median()
df_product['용량_ml'] = df_product['용량_ml'].fillna(vol_median)

print(f"용량_ml 파싱 완료: 중앙값={vol_median:.1f}mL  범위={df_product['용량_ml'].min():.1f}~{df_product['용량_ml'].max():.1f}mL")

# 가격대 + 카테고리 + 용량 (평점 제외)
df_cat = df_product[['product_id', '카테고리중', '공급가', '상품명', '용량_ml']].copy()
df_cat['price_tier'] = pd.cut(
    df_cat['공급가'],
    bins=[0, 30, 71, 425],
    labels=['low', 'mid', 'high']
)
price_dummies = pd.get_dummies(df_cat['price_tier'], prefix='price').astype(int)
cat_dummies   = pd.get_dummies(df_cat['카테고리중'],  prefix='cat').astype(int)
df_cat = pd.concat([df_cat[['product_id', '상품명', '용량_ml']], price_dummies, cat_dummies], axis=1)

# 전체 병합
df_model = df_y[['product_id', 'y_final']].merge(df_feat, on='product_id', how='inner')
df_model = df_model.merge(df_cat, on='product_id', how='left')
df_model = df_model.dropna(subset=[c for c in df_model.columns if c != '상품명'])
df_model['상품명'] = df_model['상품명'].fillna('')

# 상품명 TF-IDF (50개, 1~2gram)
tfidf = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df_model['상품명'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f"tfidf_{f}" for f in tfidf.get_feature_names_out()],
    index=df_model.index
)

X_cols = [c for c in df_model.columns if c not in ['product_id', 'y_final', '상품명']]
X = pd.concat([df_model[X_cols], tfidf_df], axis=1)
y = df_model['y_final']

print(f"\n최종 데이터셋: {df_model.shape[0]}개 제품")
print(f"\n피처 구성 ({len(X.columns)}개 총):")
print(f"  K뷰티 성분 이진:      {len(k_feat_cols)}개")
print(f"  미국 트렌드 성분 이진: {len(us_feat_cols)}개")
print(f"  합산 비율:             2개")
print(f"  성분 개수:             1개")
print(f"  가격대 (원핫):         {len([c for c in X_cols if c.startswith('price_')])}개")
print(f"  카테고리 (원핫):       {len([c for c in X_cols if c.startswith('cat_')])}개")
print(f"  용량 (mL):             1개")
print(f"  상품명 TF-IDF:         {tfidf_df.shape[1]}개")

용량_ml 파싱 완료: 중앙값=50.0mL  범위=0.3~1981.2mL

최종 데이터셋: 1767개 제품

피처 구성 (82개 총):
  K뷰티 성분 이진:      9개
  미국 트렌드 성분 이진: 10개
  합산 비율:             2개
  성분 개수:             1개
  가격대 (원핫):         3개
  카테고리 (원핫):       6개
  용량 (mL):             1개
  상품명 TF-IDF:         50개


### 모델 학습 비교: 회귀 vs 분류

### 전처리 / 정규화 / 인코딩 확인

In [47]:
from sklearn.preprocessing import StandardScaler

print("=== 결측치 확인 ===")
missing = X.isnull().sum()
print(missing[missing > 0] if missing.any() else "결측치 없음 ✅")

# total_ingredient_count, 용량_ml → StandardScaler
# 나머지 (이진, 비율, TF-IDF) → 이미 0~1
continuous_cols = ['total_ingredient_count', '용량_ml']

print(f"\n=== 피처 스케일 현황 ===")
for col in continuous_cols:
    print(f"  {col:<25} {X[col].min():.1f} ~ {X[col].max():.1f}  → StandardScaler 적용")
print(f"  나머지 {len(X.columns)-2}개 피처         0 ~ 1  → 그대로")

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[continuous_cols] = scaler.fit_transform(X[continuous_cols])

print(f"\n✅ X_scaled 준비 완료  ({X_scaled.shape[1]}개 피처)")

=== 결측치 확인 ===
결측치 없음 ✅

=== 피처 스케일 현황 ===
  total_ingredient_count    1.0 ~ 116.0  → StandardScaler 적용
  용량_ml                     0.3 ~ 1981.2  → StandardScaler 적용
  나머지 80개 피처         0 ~ 1  → 그대로

✅ X_scaled 준비 완료  (82개 피처)


In [49]:
# ─────────────────────────────────────────────────────
# 2. 분류 (y_final → 3클래스: 하위30%=0, 중간40%=1, 상위30%=2)
# ─────────────────────────────────────────────────────
y_class = pd.cut(
    y,
    bins=[0, y.quantile(0.3), y.quantile(0.7), 1.0],
    labels=[0, 1, 2],
    include_lowest=True
).astype(int)

print(f"클래스 분포:")
print(f"  0 (트렌드 비적합): {(y_class==0).sum()}개")
print(f"  1 (보통):          {(y_class==1).sum()}개")
print(f"  2 (트렌드 적합):   {(y_class==2).sum()}개\n")

clf_models = {
    'LogisticRegression': LogisticRegression(max_iter=3000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':            XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0),
    'LightGBM':           LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

clf_results = {}
for name, model in clf_models.items():
    cv = cross_validate(model, X_scaled, y_class, cv=skf,
                        scoring={'accuracy': 'accuracy',
                                 'f1_macro': 'f1_macro'},
                        return_train_score=False)
    clf_results[name] = {
        'Accuracy':  cv['test_accuracy'].mean(),
        'F1(macro)': cv['test_f1_macro'].mean(),
    }

print("=" * 50)
print("[ 분류 결과 ] StratifiedKFold(n_splits=5)")
print("=" * 50)
print(f"{'모델':<22} {'Accuracy':>10} {'F1(macro)':>10}")
print("-" * 50)
for name, res in clf_results.items():
    print(f"{name:<22} {res['Accuracy']:>10.4f} {res['F1(macro)']:>10.4f}")
print()
print("* Accuracy: 전체 정확도  |  F1(macro): 클래스별 균형 정확도")
print("* 목표 성능: Accuracy 0.7 이상")

클래스 분포:
  0 (트렌드 비적합): 530개
  1 (보통):          707개
  2 (트렌드 적합):   530개

[ 분류 결과 ] StratifiedKFold(n_splits=5)
모델                       Accuracy  F1(macro)
--------------------------------------------------
LogisticRegression         0.4318     0.4201
RandomForest               0.4216     0.4164
XGBoost                    0.4267     0.4174
LightGBM                   0.4250     0.4210

* Accuracy: 전체 정확도  |  F1(macro): 클래스별 균형 정확도
* 목표 성능: Accuracy 0.7 이상


### 이진 분류 (상위 30% vs 하위 30%, 중간 제외)

In [50]:
# 중간 클래스(1) 제외 → 하위 30%=0, 상위 30%=1 만 사용
binary_mask = y_class != 1
X_bin = X_scaled[binary_mask]
y_bin = (y_class[binary_mask] == 2).astype(int)  # 상위30%=1, 하위30%=0

print(f"이진 분류 데이터: {len(y_bin)}개 (중간 {(~binary_mask).sum()}개 제외)")
print(f"  0 (트렌드 비적합): {(y_bin==0).sum()}개")
print(f"  1 (트렌드 적합):   {(y_bin==1).sum()}개\n")

bin_models = {
    'LogisticRegression': LogisticRegression(max_iter=3000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':            XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0),
    'LightGBM':           LGBMClassifier(n_estimators=200, learning_rate=0.05, random_state=42, verbose=-1),
}

skf_bin = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

bin_results = {}
for name, model in bin_models.items():
    cv = cross_validate(model, X_bin, y_bin, cv=skf_bin,
                        scoring={'accuracy': 'accuracy',
                                 'f1':       'f1',
                                 'roc_auc':  'roc_auc'},
                        return_train_score=False)
    bin_results[name] = {
        'Accuracy': cv['test_accuracy'].mean(),
        'F1':       cv['test_f1'].mean(),
        'ROC-AUC':  cv['test_roc_auc'].mean(),
    }

print("=" * 55)
print("[ 이진 분류 결과 ] StratifiedKFold(n_splits=5)")
print("=" * 55)
print(f"{'모델':<22} {'Accuracy':>10} {'F1':>8} {'ROC-AUC':>10}")
print("-" * 55)
for name, res in bin_results.items():
    print(f"{name:<22} {res['Accuracy']:>10.4f} {res['F1']:>8.4f} {res['ROC-AUC']:>10.4f}")
print()
print("* ROC-AUC: 0.5=랜덤, 1.0=완벽  |  0.6 이상이면 의미 있음")

이진 분류 데이터: 1060개 (중간 707개 제외)
  0 (트렌드 비적합): 530개
  1 (트렌드 적합):   530개

[ 이진 분류 결과 ] StratifiedKFold(n_splits=5)
모델                       Accuracy       F1    ROC-AUC
-------------------------------------------------------
LogisticRegression         0.6623   0.6598     0.7001
RandomForest               0.6575   0.6625     0.6970
XGBoost                    0.6481   0.6566     0.6988
LightGBM                   0.6349   0.6392     0.6907

* ROC-AUC: 0.5=랜덤, 1.0=완벽  |  0.6 이상이면 의미 있음


### 하이퍼파라미터 튜닝 (극단 30%, LogisticRegression)

In [54]:
from sklearn.model_selection import GridSearchCV

# 극단 30% 데이터 재확인
binary_mask = y_class != 1
X_bin = X_scaled[binary_mask]
y_bin = (y_class[binary_mask] == 2).astype(int)
print(f"튜닝 데이터: {len(y_bin)}개  (0: {(y_bin==0).sum()}, 1: {(y_bin==1).sum()})\n")

# 탐색할 파라미터 조합
param_grid = {
    'C':       [0.001, 0.01, 0.1, 1, 10, 100],   # 규제 강도 (작을수록 강한 규제)
    'penalty': ['l1', 'l2'],                        # 규제 방식
    'solver':  ['liblinear', 'saga'],               # l1/l2 둘 다 지원
}

lr = LogisticRegression(max_iter=3000, random_state=42)
skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    lr, param_grid, cv=skf_tune,
    scoring='roc_auc',
    n_jobs=-1, verbose=0
)
grid_search.fit(X_bin, y_bin)

print("=== 하이퍼파라미터 튜닝 결과 ===")
print(f"최적 파라미터: {grid_search.best_params_}")
print(f"최적 ROC-AUC:  {grid_search.best_score_:.4f}  (튜닝 전: 0.7010)\n")

# 상위 5개 조합 출력
results_df = pd.DataFrame(grid_search.cv_results_)
top5 = results_df[['param_C','param_penalty','param_solver','mean_test_score']]\
         .sort_values('mean_test_score', ascending=False).head(5)
top5.columns = ['C', 'penalty', 'solver', 'ROC-AUC']
print("상위 5개 조합:")
print(top5.to_string(index=False))

튜닝 데이터: 1060개  (0: 530, 1: 530)

=== 하이퍼파라미터 튜닝 결과 ===
최적 파라미터: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
최적 ROC-AUC:  0.7005  (튜닝 전: 0.7010)

상위 5개 조합:
   C penalty    solver  ROC-AUC
10.0      l2 liblinear 0.700463
10.0      l2      saga 0.700427
 1.0      l2 liblinear 0.700107
 1.0      l2      saga 0.700089
10.0      l1      saga 0.699840


c:\Users\asiae\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


### 최종 모델 학습 및 저장

In [56]:
import pickle, os

OUTPUT_DIR = r"C:\workspace\finalproject\data\model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 극단 30% 데이터로 최종 모델 학습
binary_mask = y_class != 1
X_bin_final = X_scaled[binary_mask]
y_bin_final = (y_class[binary_mask] == 2).astype(int)

final_model = LogisticRegression(C=10, max_iter=3000, random_state=42)
final_model.fit(X_bin_final, y_bin_final)

# 저장
pickle.dump(final_model, open(f"{OUTPUT_DIR}/trend_model.pkl",   "wb"))
pickle.dump(tfidf,       open(f"{OUTPUT_DIR}/tfidf.pkl",         "wb"))
pickle.dump(scaler,      open(f"{OUTPUT_DIR}/scaler.pkl",        "wb"))

# 피처 목록 저장 (한국 상품 예측 시 동일 피처 순서 맞추기 위해)
feature_cols = list(X_scaled.columns)
pickle.dump(feature_cols, open(f"{OUTPUT_DIR}/feature_cols.pkl", "wb"))

print("=== 저장 완료 ===")
print(f"  trend_model.pkl  → 최종 LogisticRegression 모델")
print(f"  tfidf.pkl        → 상품명 TF-IDF 벡터라이저")
print(f"  scaler.pkl       → StandardScaler (성분수, 용량)")
print(f"  feature_cols.pkl → 피처 목록 ({len(feature_cols)}개)")
print(f"\n저장 경로: {OUTPUT_DIR}")

=== 저장 완료 ===
  trend_model.pkl  → 최종 LogisticRegression 모델
  tfidf.pkl        → 상품명 TF-IDF 벡터라이저
  scaler.pkl       → StandardScaler (성분수, 용량)
  feature_cols.pkl → 피처 목록 (82개)

저장 경로: C:\workspace\finalproject\data\model_output


### 한국 상품 예측 함수

In [57]:
def predict_trend_score(
    product_name: str,       # 영문 번역된 상품명
    ingredients: list,       # 영문 성분 리스트
    price: float,            # 가격 (USD)
    category: str,           # 카테고리 (Moisturizers/Treatments/Cleansers/Eye Care/Masks/Sunscreen)
    volume_ml: float = 50.0  # 용량 (mL), 모를 경우 기본값 50mL
) -> dict:
    """
    한국 뷰티 상품의 미국 시장 트렌드 적합도 점수를 예측합니다.

    Returns:
        score      : 0~100점 (높을수록 미국 트렌드 적합)
        label      : 트렌드 적합 / 보통 / 트렌드 비적합
        probability: 상위 30% 적합 확률 (0~1)
    """
    ing_text = ' '.join(ingredients).lower()

    # ── K뷰티 성분 이진 피처 ──
    k_keys = [k for k in K_BEAUTY_INGREDIENTS if k != 'snail']
    k_feats = {f"k_{k.replace(' ','_')}": int(k in ing_text) for k in k_keys}

    # ── 미국 트렌드 성분 이진 피처 ──
    us_feats = {f"us_{k.replace(' ','_')}": int(k in ing_text) for k in US_TREND_INGREDIENTS}

    # ── 합산 비율 ──
    k_beauty_ratio = sum(k_feats.values()) / len(k_keys)
    us_trend_ratio = sum(us_feats.values()) / len(US_TREND_INGREDIENTS)

    # ── 가격대 ──
    price_low  = int(price <= 30)
    price_mid  = int(30 < price <= 71)
    price_high = int(price > 71)

    # ── 카테고리 ──
    all_cats = ['Cleansers', 'Eye Care', 'Masks', 'Moisturizers', 'Sunscreen', 'Treatments']
    cat_feats = {f"cat_{c}": int(c == category) for c in all_cats}

    # ── 연속형 피처 (스케일링) ──
    total_ingredient_count = len(ingredients)
    cont_scaled = scaler.transform([[total_ingredient_count, volume_ml]])[0]

    # ── TF-IDF ──
    tfidf_vec = tfidf.transform([product_name]).toarray()[0]
    tfidf_feats = {f"tfidf_{f}": v for f, v in zip(tfidf.get_feature_names_out(), tfidf_vec)}

    # ── 피처 DataFrame 조립 (학습 시 순서 동일하게) ──
    row = {**k_feats, **us_feats,
           'total_ingredient_count': cont_scaled[0],
           'k_beauty_ratio': k_beauty_ratio,
           'us_trend_ratio': us_trend_ratio,
           '용량_ml': cont_scaled[1],
           'price_low': price_low, 'price_mid': price_mid, 'price_high': price_high,
           **cat_feats, **tfidf_feats}

    df_input = pd.DataFrame([row])[feature_cols]

    prob  = final_model.predict_proba(df_input)[0][1]
    score = round(prob * 100, 1)

    if score >= 60:
        label = "트렌드 적합 ✅"
    elif score >= 40:
        label = "보통 🔶"
    else:
        label = "트렌드 비적합 ❌"

    return {'score': score, 'label': label, 'probability': round(prob, 4)}


# ── 예시: 한국 상품 예측 ──────────────────────────────────
result = predict_trend_score(
    product_name = "Niacinamide Brightening Serum with Centella",
    ingredients  = ["Niacinamide", "Centella Asiatica Extract", "Hyaluronic Acid",
                    "Adenosine", "Glycerin", "Panthenol"],
    price        = 25.0,
    category     = "Treatments",
    volume_ml    = 30.0
)

print("=== 예측 결과 ===")
print(f"  트렌드 적합도 점수: {result['score']}점")
print(f"  판정:               {result['label']}")
print(f"  상위 30% 확률:      {result['probability']}")

=== 예측 결과 ===
  트렌드 적합도 점수: 38.5점
  판정:               트렌드 비적합 ❌
  상위 30% 확률:      0.3848


c:\Users\asiae\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
